# Pretrain a 1B-parameter language model on two T4 GPUs

Select **GPU T4 ×2** in Kaggle. Attach the FineWeb-D1 token dataset. For a continuation, also attach the previous notebook output containing a complete checkpoint.

The trainer is in `train/train_1b_dual_t4.py`. This notebook checks the GPUs, writes a Kaggle-specific config under `/kaggle/working`, and starts the launcher.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() == 2, "Select Kaggle GPU T4 x2"


## Configure

Set `REPO` in the next cell to the repository location. It can be a writable clone under `/kaggle/working` or a read-only dataset mount under `/kaggle/input`. The runtime config is copied to `/kaggle/working/config_1b_t4.json`; the notebook does not edit the repository copy.

Leave `resume_from` empty to let the trainer search mounted Kaggle inputs for the highest checkpoint whose saved configuration matches this run. Keep the checkpoint together with the data and tokenizer used to create it.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

# Change this path if the repository is mounted as a Kaggle dataset.
REPO = Path("/kaggle/working/pretrain-1b-on-2xT4")
repo_config = REPO / "train/config_1b_t4.json"
if not repo_config.is_file():
    raise FileNotFoundError(f"Set REPO to the repository root; not found: {repo_config}")

cfg = json.loads(repo_config.read_text())
cfg["run_root"] = "/kaggle/working/educational_dense_1b_continue"
cfg["fineweb_root"] = "/kaggle/input/datasets/vijayendrad/fineweb-d1/sca_fineweb_2025_26_clean_llama32_1p5b"
cfg["resume_from"] = ""
runtime_config = Path("/kaggle/working/config_1b_t4.json")
runtime_config.write_text(json.dumps(cfg, indent=2) + "\n")
os.environ["DENSE1B_CONFIG"] = str(runtime_config)
print(json.dumps(cfg, indent=2))


In [ ]:
subprocess.run([sys.executable, str(REPO / "train/launch_kaggle.py")], check=True)
